# 04 — Head-to-head model comparison

**Goal**: the payoff. Load both trained checkpoints and produce the full visual report — training curves, confusion matrices, ROC, and a misclassification gallery.

All plotting routines come from `src/evaluation/visualizations.py` so this notebook stays declarative.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.data import DogsVsCatsDataModule, download_dataset
from src.models import CNNModel, LogRegModel
from src.evaluation import (
    get_predictions,
    plot_training_curves,
    plot_confusion_matrices,
    plot_roc_and_probs,
    plot_misclassified,
)
from src.utils import load_config, set_seed

set_seed(42)

## 1. Load checkpoints

Adjust the paths to whatever your runs produced. The Lightning checkpoint filenames include the validation accuracy, so they're easy to identify.

In [ ]:
LOGREG_CKPT = '../checkpoints/logreg-best.ckpt'
CNN_CKPT    = '../checkpoints/cnn-best.ckpt'

logreg = LogRegModel.load_from_checkpoint(LOGREG_CKPT)
cnn    = CNNModel.load_from_checkpoint(CNN_CKPT)
print('Both models loaded.')

## 2. Re-build the data module (same split, same seed)

In [ ]:
cfg = load_config('../configs/base.yaml')
data_root = download_dataset(cfg['data']['dataset_name'])

dm = DogsVsCatsDataModule(
    data_dir=data_root,
    batch_size=cfg['data']['batch_size'],
    img_size=cfg['data']['img_size'],
    num_workers=cfg['data']['num_workers'],
    seed=cfg['experiment']['seed'],
)
dm.setup()
test_loader = dm.test_dataloader()

## 3. Predictions on the test set

In [ ]:
preds_lr  = get_predictions(logreg, test_loader)
preds_cnn = get_predictions(cnn,    test_loader)
predictions = {'logreg': preds_lr, 'cnn': preds_cnn}

print(f'LogReg test predictions: {len(preds_lr.preds)} samples')
print(f'CNN    test predictions: {len(preds_cnn.preds)} samples')

## 4. Figure 1 — Training curves (loss, accuracy, overfit gap, final metrics)

Requires the histories from training. We read them from disk.

In [ ]:
metrics_dir = Path('../reports/metrics')
with open(metrics_dir / 'logreg_test.json') as f: lr_test  = json.load(f)
with open(metrics_dir / 'cnn_test.json')    as f: cnn_test = json.load(f)

# Histories are part of the LightningModule's internal state, persisted across the .fit()/.test() call.
# When loading from checkpoint they need to have been recorded explicitly — for this comparison
# notebook we assume they are available on the loaded model instances.
histories = {'logreg': logreg.history, 'cnn': cnn.history}

plot_training_curves(
    histories,
    {'logreg': lr_test, 'cnn': cnn_test},
    '../reports/figures/fig1_training_curves.png',
)

## 5. Figure 2 — Confusion matrices

In [ ]:
plot_confusion_matrices(
    predictions, dm.classes,
    '../reports/figures/fig2_confusion.png'
)

## 6. Figure 3 — ROC + probability distributions

In [ ]:
plot_roc_and_probs(
    predictions, dm.classes,
    '../reports/figures/fig3_roc_probs.png'
)

## 7. Figure 4 — Misclassified gallery

Look at *which* images each model gets wrong. Often the LogReg failures are the visually "average" images; the CNN failures are the unusual ones — strange poses, low contrast, partial occlusion.

In [ ]:
plot_misclassified(logreg, test_loader, dm.classes,
                   'Logistic Regression',
                   '../reports/figures/fig4_errors_logreg.png')
plot_misclassified(cnn,    test_loader, dm.classes,
                   'CNN',
                   '../reports/figures/fig4_errors_cnn.png')

## 8. Disagreement analysis

On how many test images do the two models *agree*? Where they disagree, who's right?

In [ ]:
import numpy as np

agree = preds_lr.preds == preds_cnn.preds
disagree = ~agree

lr_right_on_disagree  = (preds_lr.preds[disagree]  == preds_lr.labels[disagree]).mean()
cnn_right_on_disagree = (preds_cnn.preds[disagree] == preds_cnn.labels[disagree]).mean()

print(f'Models agree on {agree.mean()*100:.1f}% of test images')
print(f'Where they disagree:')
print(f'  - LogReg is right: {lr_right_on_disagree*100:.1f}%')
print(f'  - CNN    is right: {cnn_right_on_disagree*100:.1f}%')

## 9. Conclusion

The comparison is unambiguous: at this resolution, the CNN's inductive bias is *the* thing that makes the task tractable. A linear model cannot meaningfully separate cats from dogs by raw pixel intensity, and adding more linear capacity wouldn't help — the failure mode is **representational**, not capacity-bounded.

**Where to go next?** See the [roadmap in the README](../README.md#-roadmap--improvements). The most impactful single change would be transfer learning from a pretrained ImageNet backbone.